<a href="https://colab.research.google.com/github/AndrewIsk/clinical-routing-agent/blob/main/LLM_Model_%2B_RAG_%2B_AWS_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [18]:
df = pd.read_csv("https://raw.githubusercontent.com/socd06/medical-nlp/master/data/mtsamples.csv")
#df

In [19]:
print(len(df))
print(df.columns)
print(len(df['medical_specialty'].unique()))

4999
Index(['Unnamed: 0', 'description', 'medical_specialty', 'sample_name',
       'transcription', 'keywords'],
      dtype='object')
40


In [20]:
df[df['medical_specialty'].str.strip() == 'Surgery']['sample_name'].head(15)
df[df['medical_specialty'].str.strip() == 'General Medicine']['sample_name'].head(15)

,sample_name
19,Airway Compromise & Foreign Body - ER Visit
3200,Viral Syndrome - ER Visit
3201,URI - SOAP
3202,Weight Loss Evaluation
3203,Trouble breathing
3205,Trouble Breathing - H&P
3206,Toothache - ER Visit
3207,Tongue Swelling
3208,Thrombocytopenia - SOAP Note
3209,Sleepiness - Consult


In [21]:
keep = ['Cardiovascular / Pulmonary', 'Orthopedic', 'Radiology',
 'General Medicine', 'Gastroenterology', 'Neurology', 'Obstetrics / Gynecology', 'Urology', 'ENT - Otolaryngology', 'Neurosurgery', 'Hematology - Oncology',
 'Ophthalmology', 'Nephrology', 'Pediatrics - Neonatal', 'Pain Management', 'Psychiatry / Psychology', 'Podiatry'
]

df['medical_specialty'] = df['medical_specialty'].str.strip()
df = df.dropna(subset=['transcription'])
df = df.drop(df[df['transcription'].str.len() <= 30].index)
df = df[df['medical_specialty'].isin(keep)]
df = df.drop_duplicates(['transcription']).reset_index(drop=True)
print(len(df))
print(df['medical_specialty'].nunique())
print(df['medical_specialty'].value_counts().tail())
#df

2120
17
medical_specialty
Psychiatry / Psychology    53
Pain Management            51
Nephrology                 47
Neurosurgery               46
Podiatry                   39
Name: count, dtype: int64


In [22]:
train_df, test_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['medical_specialty'])
train_df, val_df = train_test_split(train_df, test_size=0.1765, random_state=42, stratify=train_df['medical_specialty'])

train_df.to_csv('train.csv', index=False)
val_df.to_csv('validate.csv', index=False)
test_df.to_csv('test.csv', index=False)

In [23]:
print(train_df['medical_specialty'].value_counts(normalize=True).tail())
print(val_df['medical_specialty'].value_counts(normalize=True).tail())
print(test_df['medical_specialty'].value_counts(normalize=True).tail())

medical_specialty
Psychiatry / Psychology    0.024949
Pain Management            0.023601
Nephrology                 0.022252
Neurosurgery               0.021578
Podiatry                   0.018206
Name: proportion, dtype: float64
medical_specialty
Psychiatry / Psychology    0.025078
Pain Management            0.025078
Nephrology                 0.021944
Neurosurgery               0.021944
Podiatry                   0.018809
Name: proportion, dtype: float64
medical_specialty
Psychiatry / Psychology    0.025157
Pain Management            0.025157
Neurosurgery               0.022013
Nephrology                 0.022013
Podiatry                   0.018868
Name: proportion, dtype: float64


In [24]:
label2id = {'Cardiovascular / Pulmonary': 0,
            'Orthopedic': 1,
            'Radiology': 2,
            'General Medicine': 3,
            'Podiatry': 4,
            'Gastroenterology': 5,
            'Neurology': 6,
            'Obstetrics / Gynecology': 7,
            'Urology': 8,
            'Neurosurgery': 9,
            'Hematology - Oncology': 10,
            'Ophthalmology': 11,
            'Nephrology': 12,
            'Pediatrics - Neonatal': 13,
            'Pain Management': 14,
            'Psychiatry / Psychology': 15,
            'ENT - Otolaryngology': 16}

id2label = {0: 'Cardiovascular / Pulmonary',
            1: 'Orthopedic',
            2: 'Radiology',
            3: 'General Medicine',
            4: 'Podiatry',
            5: 'Gastroenterology',
            6: 'Neurology',
            7: 'Obstetrics / Gynecology',
            8: 'Urology',
            9: 'Neurosurgery',
            10: 'Hematology - Oncology',
            11: 'Ophthalmology',
            12: 'Nephrology',
            13: 'Pediatrics - Neonatal',
            14: 'Pain Management',
            15: 'Psychiatry / Psychology',
            16: 'ENT - Otolaryngology'}

In [25]:
print(len(label2id), len(id2label))
print(all(id2label[label2id[k]] == k for k in label2id))

17 17
True


In [26]:
train_df['label'] = train_df['medical_specialty'].map(label2id)
val_df['label'] = val_df['medical_specialty'].map(label2id)
test_df['label'] = test_df['medical_specialty'].map(label2id)

print(train_df['label'].isna().sum())
print(val_df['label'].isna().sum())
print(test_df['label'].isna().sum())

0
0
0


In [27]:
train_df[['medical_specialty', 'label']].head()

,medical_specialty,label
994,Obstetrics / Gynecology,7
2079,Cardiovascular / Pulmonary,0
1032,Obstetrics / Gynecology,7
596,Pain Management,14
1670,Gastroenterology,5


In [28]:
from transformers import AutoTokenizer
from datasets import Dataset
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from transformers import TrainingArguments
from transformers import Trainer
from transformers import DataCollatorWithPadding
from google.colab import drive

drive.mount('/content/drive')

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
train_ds = Dataset.from_pandas(train_df[['transcription', 'label']].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[['transcription', 'label']].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[['transcription', 'label']].reset_index(drop=True))

print(train_ds)
#train_df['transcription'].apply(lambda x: len(tokenizer(x)['input_ids'])).describe()

Mounted at /content/drive


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Dataset({
    features: ['transcription', 'label'],
    num_rows: 1483
})


In [29]:
def tokenize(batch):
  return tokenizer(batch['transcription'], truncation=True, max_length=512)
train_ds_tokenize = train_ds.map(tokenize, batched=True)
val_ds_tokenize = val_ds.map(tokenize, batched=True)
test_ds_tokenize = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/1483 [00:00<?, ? examples/s]

Map:   0%|          | 0/319 [00:00<?, ? examples/s]

Map:   0%|          | 0/318 [00:00<?, ? examples/s]

In [30]:
print(train_ds_tokenize)
print(val_ds_tokenize)
print(test_ds_tokenize)

print(train_ds_tokenize[0]['input_ids'][:10])
print(len(train_ds_tokenize[0]['input_ids']))

print(train_ds_tokenize[1]['input_ids'][0])
print(train_ds_tokenize[1]['input_ids'][-1])
print(train_ds_tokenize[0]['input_ids'][-1])

Dataset({
    features: ['transcription', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1483
})
Dataset({
    features: ['transcription', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 319
})
Dataset({
    features: ['transcription', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 318
})
[101, 2695, 25918, 8082, 2154, 1001, 1015, 1010, 2561, 21419]
210
101
102
102


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
  "distilbert-base-uncased",
  num_labels=17,
  id2label=id2label,
  label2id=label2id
)

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  pred = np.argmax(logits, axis=1)
  return {
      "accuracy": accuracy_score(labels, pred),
      "f1_macro": f1_score(labels, pred, average='macro')
  }

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/specialty-classifier",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_tokenize,
    eval_dataset=val_ds_tokenize,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

trainer.train()

trainer.save_model("/content/drive/MyDrive/specialty-classifier/final")
tokenizer.save_pretrained("/content/drive/MyDrive/specialty-classifier/final")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(load

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.447575,0.620690,0.421086
2,1.928693,0.980640,0.711599,0.599186


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.447575,0.620690,0.421086
2,1.928693,0.980640,0.711599,0.599186
3,0.823487,0.812633,0.749216,0.667820


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
import os
os.listdir("./specialty-classifier")